In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("amazon_reviews.csv")

df = df[["reviewText", "overall"]].dropna()

sentiment_map = {1.0: 0, 2.0: 0, 3.0: 1, 4.0: 2, 5.0: 2}
df["label"] = df["overall"].map(sentiment_map)
df = df.rename(columns={"reviewText": "text"}).dropna(subset=["label"])
df["label"] = df["label"].astype(int)

samples = [
    df[df["label"] == c].sample(
        min(len(df[df["label"] == c]), 500), random_state=42
    )
    for c in [0, 1, 2]
]
df = (
    pd.concat(samples, ignore_index=True)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

train_df, test_df = train_test_split(
    df[["text", "label"]], test_size=0.2, random_state=42, stratify=df["label"]
)

print(f"Train samples: {len(train_df)}, Test samples: {len(test_df)}")
print("\nClass distribution:\n", train_df["label"].value_counts())

Train samples: 583, Test samples: 146

Class distribution:
 label
2    400
0    108
1     75
Name: count, dtype: int64


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

words = [w for t in train_df["text"] for w in t.lower().split()]
vocab = {
    w: i + 2
    for i, (w, _) in enumerate(pd.Series(words).value_counts()[:4000].items())
}

def encode(text, maxlen=60):
    ids = [vocab.get(w, 1) for w in text.lower().split()[:maxlen]]
    return ids + [0] * (maxlen - len(ids))


class TextData(Dataset):

    def __init__(self, data):
        self.x = torch.tensor(
            [encode(t) for t in data["text"]], dtype=torch.long
        )
        self.y = torch.tensor(data["label"].values, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


train_loader = DataLoader(TextData(train_df), batch_size=32, shuffle=True)
test_loader = DataLoader(TextData(test_df), batch_size=64)


class SimpleLSTM(nn.Module):

    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(4002, 32, padding_idx=0)
        self.lstm = nn.LSTM(32, 32, batch_first=True)
        self.fc = nn.Linear(32, 3)

    def forward(self, x):
        _, (hn, _) = self.lstm(self.emb(x))
        return self.fc(hn[-1])


lstm = SimpleLSTM().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lstm.parameters(), lr=0.003)

lstm.train()
for epoch in range(4):
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        criterion(lstm(x_b), y_b).backward()
        optimizer.step()

lstm.eval()
correct = sum(
    (lstm(x.to(device)).argmax(1) == y.to(device)).sum().item()
    for x, y in test_loader
)
print(f"LSTM Test Accuracy: {correct / len(test_df):.2%}")

LSTM Test Accuracy: 68.49%


In [6]:
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize(batch):
    return tokenizer(
        batch["text"], truncation=True, padding="max_length", max_length=64
    )


ds_train = Dataset.from_pandas(train_df).map(tokenize, batched=True)
ds_test = Dataset.from_pandas(test_df).map(tokenize, batched=True)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=3
)

args = TrainingArguments(
    output_dir="./bert_simple",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    eval_strategy="no",  
    save_strategy="no",
    logging_steps=20,
)

trainer = Trainer(
    model=bert_model, args=args, train_dataset=ds_train, eval_dataset=ds_test
)
trainer.train()

preds = trainer.predict(ds_test)
bert_acc = (preds.predictions.argmax(-1) == preds.label_ids).mean()
print(f"DistilBERT Test Accuracy: {bert_acc:.2%}")

Map:   0%|          | 0/583 [00:00<?, ? examples/s]

Map:   0%|          | 0/146 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
20,0.883378


DistilBERT Test Accuracy: 68.49%


In [7]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    logits = bert_model(**inputs.to(bert_model.device)).logits
    label_id = logits.argmax().item()
    labels = ["Negative", "Neutral", "Positive"]
    return labels[label_id]


print(predict_sentiment("Broke after two days of use, waste of money."))
print(predict_sentiment("Average product, works as expected."))
print(predict_sentiment("Works great! Really impressed with the quality."))

Positive
Positive
Positive
